# 🍽️ Restaurant Recommendation Bot: RAG Demo
**A Step-by-Step Retrieval-Augmented Generation (RAG) Demonstration for Google Colab**

Welcome to this interactive demonstration of a **Domain-Specific Chatbot** built using RAG. This bot recommends restaurants based on:
* **Cuisines & Signature Menus**
* **Price Tier & Budget** (`$`, `$$`, `$$$`, `$$$$`)
* **Dietary Needs** (Vegan, Gluten-Free, Halal, Nut-Free, Jain)
* **Neighborhoods & Locations** (Downtown, Waterfront, Midtown, Arts District, etc.)
* **Atmosphere & Occasion** (Romantic date night, quiet business dinner, casual kid-friendly, late-night)

---

### 🧠 How Retrieval-Augmented Generation (RAG) Works:
1. **Knowledge Base**: Curated restaurant profiles with rich structured and descriptive attributes.
2. **Embeddings & Vector Store**: Text is converted into semantic vector embeddings and indexed in **ChromaDB**.
3. **Semantic Retrieval**: For any user question, the vector store retrieves the most relevant restaurant profiles.
4. **Grounded LLM Generation**: **Google Gemini (1.5 Flash)** synthesizes a customized, transparent recommendation citing exact sources—without hallucinating.


In [ ]:
# 📦 Install required libraries
# In Google Colab, run this cell to install LangChain, Gemini API, ChromaDB, and Sentence-Transformers
!pip install -q -U langchain langchain-community langchain-google-genai chromadb sentence-transformers
print("✅ Libraries installed successfully!")


In [ ]:
# 🔑 Setup Google Gemini API Key
import os
import getpass

# Method 1: Fetch from Google Colab Secrets (Recommended)
# Click the 'Key' icon on the left sidebar in Colab -> Add secret 'GEMINI_API_KEY'
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

# Method 2: Interactive prompt fallback if not found in secrets
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key (get one free at aistudio.google.com): ")

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
print("✅ Gemini API Key configured successfully!")


## 📚 Step 1: Ingesting the Restaurant Knowledge Base

In standard LLM chatbots, models often hallucinate non-existent restaurants or outdated hours. In RAG, we supply our own verified catalog of documents.

Below, we define a curated catalog of restaurants covering varied cuisines, price points, dietary certifications, and vibes.


In [ ]:
# 📝 Define Curated Restaurant Profiles with Structured Metadata
from langchain_core.documents import Document

restaurant_documents = [
    Document(
        page_content="""Restaurant Name: Bella Italia Trattoria
Cuisine: Authentic Northern Italian
Neighborhood: Downtown / Old Quarter
Price Range: $$ ($20 - $35 per person)
Dietary Accommodations: Vegetarian-friendly, handmade gluten-free pasta available upon request. Not certified vegan or halal.
Atmosphere & Ambiance: Cozy, candle-lit, rustic exposed brick walls, romantic date-night ambiance with soft jazz.
Signature Dishes: Handmade Truffle Tagliolini, Risotto ai Funghi Porcini, Wood-fired Margherita D.O.P., Tiramisu della Nonna.
Features & Amenities: Heated outdoor garden patio, curated Tuscan and Chianti wine list, reservations recommended for weekend evenings.
Hours: Tuesday to Sunday 5:00 PM - 10:30 PM (Closed Mondays).""",
        metadata={"name": "Bella Italia Trattoria", "cuisine": "Italian", "neighborhood": "Downtown", "price": "$$", "rating": 4.8}
    ),
    Document(
        page_content="""Restaurant Name: Verde Botanica
Cuisine: Modern Californian Farm-to-Table & 100% Plant-Based
Neighborhood: West End / Garden District
Price Range: $$ ($18 - $30 per person)
Dietary Accommodations: 100% Vegan, 100% Organic, Nut-free & Soy-free dishes clearly labeled, dedicated Gluten-Free kitchen station.
Atmosphere & Ambiance: Bright, airy greenhouse aesthetic with living plant walls, Scandinavian oak tables, relaxed & casual.
Signature Dishes: Lion's Mane Mushroom Steak with chimichurri, Roasted Golden Beet Carpaccio, Truffled Cashew Mac & Cheese, Spirulina Smoothie Bowls.
Features & Amenities: Dog-friendly sidewalk patio, zero-waste certified, biodynamic natural wines, locally roasted fair-trade espresso.
Hours: Monday to Sunday 8:00 AM - 9:00 PM (All-day brunch & dinner).""",
        metadata={"name": "Verde Botanica", "cuisine": "Vegan / Plant-Based", "neighborhood": "West End", "price": "$$", "rating": 4.9}
    ),
    Document(
        page_content="""Restaurant Name: Ocean & Ember Prime Seafood & Grill
Cuisine: Upscale Coastal Seafood & Dry-Aged Prime Steaks
Neighborhood: Waterfront Marina
Price Range: $$$$ ($85 - $150 per person)
Dietary Accommodations: Extensive Pescatarian menu, Dairy-free & Gluten-free preparations upon request. Certified Halal wagyu available with 24-hour advance booking.
Atmosphere & Ambiance: Sophisticated, quiet luxury, panoramic marina ocean views, linen tablecloths. Perfect for executive business dinners, client entertaining, and milestone celebrations.
Signature Dishes: Chilean Sea Bass with ginger-miso glaze, 45-day Dry-Aged Tomahawk Ribeye, Chilled Alaskan King Crab Tower, Truffle Potato Purée.
Features & Amenities: Master Sommelier-curated cellar (600+ bottles), complimentary valet parking, smart-casual dress code, two private executive dining rooms seating up to 12.
Hours: Monday to Saturday 5:30 PM - 11:00 PM (Closed Sundays).""",
        metadata={"name": "Ocean & Ember Prime Seafood & Grill", "cuisine": "Seafood & Steakhouse", "neighborhood": "Waterfront", "price": "$$$$", "rating": 4.9}
    ),
    Document(
        page_content="""Restaurant Name: Taquería El Corazón
Cuisine: Authentic Mexican Street Food & Oaxacan Specialties
Neighborhood: Arts District
Price Range: $ ($10 - $18 per person)
Dietary Accommodations: Naturally gluten-free 100% yellow corn tortillas, vegetarian grilled cactus (nopales) tacos, vegan black bean tamales.
Atmosphere & Ambiance: Vibrant, high-energy, colorful Oaxacan murals, festive Latin music, family-friendly with booster seats and kids' coloring menus.
Signature Dishes: Al Pastor Tacos shaved directly from the vertical spit with grilled pineapple, Birria de Res with slow-simmered consommé dip, Handmade Queso Fundido with chorizo, Hot cinnamon churros.
Features & Amenities: Spacious outdoor string-lit courtyard, walk-in only (no reservations required), kids eat free on Tuesdays, 35+ artisan mezcals and house-infused margaritas.
Hours: Wednesday to Sunday 11:30 AM - 11:00 PM (Late-night weekend window open until 1:30 AM).""",
        metadata={"name": "Taquería El Corazón", "cuisine": "Mexican Street Food", "neighborhood": "Arts District", "price": "$", "rating": 4.7}
    ),
    Document(
        page_content="""Restaurant Name: Sakura Omakase & Robata
Cuisine: Contemporary Japanese Sushi & Binchotan Charcoal Robata
Neighborhood: Midtown
Price Range: $$$ ($60 - $100 per person)
Dietary Accommodations: Pescatarian-friendly, authentic sashimi & nigiri, gluten-free tamari soy sauce upon request. Shellfish-free accommodations available.
Atmosphere & Ambiance: Minimalist Zen interior, dark slate and Hinoki cedar wood, 12-seat open chef counter, intimate low lighting.
Signature Dishes: 14-course Chef's Seasonal Omakase, Miyazaki A5 Wagyu Robata Skewers, Bluefin Tuna Otoro flight with fresh wasabi, Yuzu Matcha Soufflé.
Features & Amenities: Curated Rare Junmai Daiginjo sake tasting flights, app reservation required 14 days in advance, private tatami booths for quiet conversation.
Hours: Tuesday to Sunday 5:30 PM - 10:30 PM.""",
        metadata={"name": "Sakura Omakase & Robata", "cuisine": "Japanese", "neighborhood": "Midtown", "price": "$$$", "rating": 4.9}
    ),
    Document(
        page_content="""Restaurant Name: Spice Route Palace
Cuisine: Royal North Indian & Mughlai Delicacies
Neighborhood: Uptown Cultural Plaza
Price Range: $$ ($22 - $40 per person)
Dietary Accommodations: 100% Certified Halal meats, dedicated Jain vegetarian menu (no onion/garlic), large selection of vegan curries, nut allergy alerts strictly documented.
Atmosphere & Ambiance: Grand royal decor with handcrafted brass lamps and silk drapery, spacious round banquet tables, warm family hospitality.
Signature Dishes: Dum Pukht Chicken Biryani sealed with dough, Dal Makhani slow-cooked for 24 hours, Butter Chicken Royale, Garlic & Rosemary Butter Naan, Saffron Pistachio Kulfi.
Features & Amenities: Ideal for large family gatherings and reunions (tables up to 16 people), weekend buffet lunch, full catering service, wheelchair accessible.
Hours: Daily Lunch 12:00 PM - 3:00 PM, Dinner 5:30 PM - 10:30 PM.""",
        metadata={"name": "Spice Route Palace", "cuisine": "Indian", "neighborhood": "Uptown", "price": "$$", "rating": 4.8}
    ),
    Document(
        page_content="""Restaurant Name: Le Petit Marché Bistro
Cuisine: Traditional French Parisian Bistro
Neighborhood: Downtown / French Quarter
Price Range: $$$ ($45 - $75 per person)
Dietary Accommodations: Traditional French culinary style (butter and cream based), excellent pescatarian dishes and vegetarian goat cheese quiche, limited vegan options.
Atmosphere & Ambiance: Nostalgic Parisian vintage charm, zinc bar counter, wicker bistro chairs, soft French accordion and chansons, intimate romantic dinner atmosphere.
Signature Dishes: French Onion Soup Gratinée with aged Gruyère, Steak Frites with house béarnaise, Duck Confit with rosemary fingerlings, Vanilla Bean Crème Brûlée.
Features & Amenities: Sidewalk street café seating, sommelier-selected French regional wines by the carafe or glass, popular Sunday Jazz Brunch.
Hours: Tuesday to Saturday 5:00 PM - 10:00 PM; Sunday Brunch 10:00 AM - 2:30 PM.""",
        metadata={"name": "Le Petit Marché Bistro", "cuisine": "French", "neighborhood": "Downtown", "price": "$$$", "rating": 4.7}
    ),
    Document(
        page_content="""Restaurant Name: Golden Lotus Noodle House
Cuisine: Northern Thai & Vietnamese Street Noodles
Neighborhood: East Village
Price Range: $ ($12 - $20 per person)
Dietary Accommodations: 100% gluten-free rice noodle soups, tofu & mushroom broth substitutions available for all bowls, dairy-free, peanut allergy notices clearly stated.
Atmosphere & Ambiance: Energetic, casual, neon sign lighting, communal wooden bench tables, fast table turnover, great for solo diners and casual meetups.
Signature Dishes: Chiang Mai Khao Soi (rich coconut curry noodle soup with crispy egg noodles), 18-Hour Simmered Beef Pho Dac Biet, Crispy Pork Belly Steamed Bao, Thai Iced Tea.
Features & Amenities: Fast service (typical prep under 8 minutes), takeout & delivery friendly, craft Asian beer selection, open late daily.
Hours: Monday to Sunday 11:00 AM - 12:00 Midnight.""",
        metadata={"name": "Golden Lotus Noodle House", "cuisine": "Thai & Vietnamese", "neighborhood": "East Village", "price": "$", "rating": 4.6}
    )
]

print(f"✅ Successfully loaded {len(restaurant_documents)} comprehensive restaurant profiles into the catalog!")


## ⚡ Step 2: Text Chunking, Vector Embeddings & ChromaDB

In this step:
1. We initialize an embedding model that converts textual descriptions into semantic vectors.
2. We store our documents inside **ChromaDB**, a fast vector database running in-memory in Colab.
3. This creates a searchable semantic index of our restaurants.


In [ ]:
# 🧠 Generate Embeddings and Index into ChromaDB Vector Store
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# We use HuggingFace's all-MiniLM-L6-v2 (runs locally in Colab, 100% free, fast, no API quota or 404 errors!)
print("⏳ Loading local embedding model (all-MiniLM-L6-v2)...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("✅ Local embedding model ready!")

# Build Chroma vector database from the documents
vector_db = Chroma.from_documents(
    documents=restaurant_documents,
    embedding=embedding_model,
    collection_name="restaurant_catalog"
)

# Create a retriever to pull top-3 most relevant restaurants per query
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print(f"✅ ChromaDB vector store successfully created with {len(restaurant_documents)} indexed restaurants!")


## 🔍 Step 3: Visualizing Semantic Retrieval (Pre-LLM Verification)

Before invoking the generative LLM, let's verify that the vector database correctly retrieves relevant restaurants based on semantic meaning, not just exact keyword matching.

*Notice how Chroma matches on concepts like "date night", "romantic", and "vegan" even if those exact words aren't in every field.*


In [ ]:
# 🧪 Test Semantic Retrieval with a sample query
test_query = "quiet cozy spot for a romantic anniversary dinner with great wine"

print(f"🔍 Test Query: '{test_query}'\n")
retrieved_test_docs = retriever.invoke(test_query)

for idx, doc in enumerate(retrieved_test_docs, 1):
    meta = doc.metadata
    print(f"Match #{idx}: {meta.get('name')} ({meta.get('cuisine')})")
    print(f"  • Neighborhood: {meta.get('neighborhood')} | Price: {meta.get('price')} | Rating: ⭐ {meta.get('rating')}")
    snippet = doc.page_content.split('Atmosphere & Ambiance:')[1].split('Signature Dishes:')[0].strip() if 'Atmosphere & Ambiance:' in doc.page_content else doc.page_content[:120]
    print(f"  • Ambiance Snippet: {snippet}\n")


## 🤖 Step 4: Grounded Prompt Engineering & Gemini LLM

Now we combine **Retrieval** with **Generation**:
1. We set up **Google Gemini 1.5 Flash** (fast, accurate, cost-effective).
2. We craft a **system prompt** that instructs the model to:
   * Act as a knowledgeable restaurant concierge.
   * Address the user's specific constraints (budget, location, diet, vibe).
   * **Strictly ground answers** in the retrieved context.
   * Cite restaurant names, neighborhoods, price ratings, and signature dishes.
   * **Never hallucinate non-existent restaurants** if the catalog doesn't have a match.


In [ ]:
# 🔗 Define the Grounded RAG Generation Pipeline
import time
import re
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# Priority list of flash models to rotate through on 429 quota exhaustion
MODELS_TO_TRY = [
    "gemini-3.8-flash",
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-3.7-flash",
]

current_model_idx = 0
llm = ChatGoogleGenerativeAI(model=MODELS_TO_TRY[current_model_idx], temperature=0.2)
print(f"✅ Initialized with primary model: {MODELS_TO_TRY[current_model_idx]}")

# Prompt Template enforcing grounding and source attribution
PROMPT_TEMPLATE = """You are "ConciergeAI", an expert restaurant concierge and dining recommendation assistant.

Your task is to provide helpful, personalized restaurant recommendations based strictly on the verified catalog context provided below.

RULES:
1. Recommend the best matching restaurant(s) from the provided context that fit the user's criteria (cuisine, budget, dietary restrictions, location, vibe).
2. Clearly highlight: Restaurant Name, Cuisine, Neighborhood, Price Tier, and why it fits their request. Mention a signature dish or amenity.
3. If the user asks for dietary accommodations (e.g. Vegan, Halal, Gluten-Free), explicitly confirm whether and how the restaurant caters to it.
4. If no restaurants in the context match the requested criteria (e.g. asking for a city or cuisine not in our catalog), politely state that our current catalog does not have a matching restaurant and offer the closest alternative from the provided context. DO NOT invent non-existent restaurants.

==============================
CATALOG CONTEXT:
{context}
==============================

USER QUESTION: {question}

CONCIERGE RECOMMENDATION:"""

prompt = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=["context", "question"])

def format_context_docs(docs):
    """Format retrieved documents into a clean context string for the prompt."""
    formatted_chunks = []
    for d in docs:
        name = d.metadata.get("name", "Restaurant")
        formatted_chunks.append(f"--- [CATALOG ENTRY: {name}] ---\n{d.page_content}")
    return "\n\n".join(formatted_chunks)

def extract_clean_text(content):
    """Extract clean readable text from LangChain / Gemini response."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join(parts)
    return str(content)

def synthesize_grounded_fallback(retrieved_docs, user_query):
    """Deterministic grounded presentation fallback if all API quotas are exhausted during meeting."""
    if not retrieved_docs:
        return "I apologize, but our current catalog does not have any restaurants matching your specific criteria. Please feel free to ask about Italian, Mexican, Vegan, Seafood, Japanese, Indian, French, or Thai dining."
    
    top = retrieved_docs[0]
    m = top.metadata
    return (
        f"Based on your request, I recommend **{m.get('name')}** from our verified catalog:\n\n"
        f"• **Restaurant:** {m.get('name')}\n"
        f"• **Cuisine:** {m.get('cuisine')}\n"
        f"• **Neighborhood:** {m.get('neighborhood')}\n"
        f"• **Price Tier:** {m.get('price')}\n"
        f"• **Details from Catalog:**\n{top.page_content}\n\n"
        f"*(Generated from retrieved ChromaDB context via grounded fallback)*"
    )

def invoke_with_retry(prompt_text, retrieved_docs, user_query):
    """Safely invoke the LLM with automatic model rotation across quotas & 503 retries."""
    global llm, current_model_idx
    
    # Try current and remaining models in the list
    for idx in range(len(MODELS_TO_TRY)):
        model_name = MODELS_TO_TRY[(current_model_idx + idx) % len(MODELS_TO_TRY)]
        try:
            active_llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.2)
            response = active_llm.invoke(prompt_text)
            current_model_idx = (current_model_idx + idx) % len(MODELS_TO_TRY)
            llm = active_llm
            return extract_clean_text(response.content)
        except Exception as e:
            err_msg = str(e)
            # Handle 429 Rate Limit / Resource Exhausted
            if "429" in err_msg or "RESOURCE_EXHAUSTED" in err_msg:
                print(f"   ℹ️ Model {model_name} quota reached (429). Rotating to next available model...")
                continue
            # Handle 503 High Demand / Server Busy
            elif "503" in err_msg or "UNAVAILABLE" in err_msg or "high demand" in err_msg:
                print(f"   ⚠️ Model {model_name} busy (503). Retrying in 2s...")
                time.sleep(2)
                try:
                    response = active_llm.invoke(prompt_text)
                    return extract_clean_text(response.content)
                except Exception:
                    continue
            else:
                # Other error, try next candidate
                continue
    
    # If all API models hit daily quota cap, use verified grounded fallback so meeting demo never fails!
    print("   🛡️ All API free-tier quotas reached for the minute. Using verified RAG catalog grounding:")
    return synthesize_grounded_fallback(retrieved_docs, user_query)

def ask_restaurant_bot(user_query: str, top_k: int = 3):
    """Full RAG execution: Retrieve -> Format -> Augment -> Generate"""
    print("=" * 80)
    print(f"💬 USER QUERY: {user_query}")
    print("=" * 80)
    
    # 1. Retrieve top matching documents
    retrieved_docs = retriever.invoke(user_query)
    
    print(f"\n📑 RETRIEVED {len(retrieved_docs)} RELEVANT RESTAURANT(S) FROM VECTOR DB:")
    for i, doc in enumerate(retrieved_docs, 1):
        m = doc.metadata
        print(f"   [{i}] {m.get('name')} | Cuisine: {m.get('cuisine')} | Loc: {m.get('neighborhood')} | Price: {m.get('price')}")
    
    # 2. Format context string
    context_str = format_context_docs(retrieved_docs)
    
    # 3. Generate response using LLM with retry & formatting
    formatted_prompt = prompt.format(context=context_str, question=user_query)
    answer_text = invoke_with_retry(formatted_prompt, retrieved_docs, user_query)
    
    print(f"\n🤖 BOT RECOMMENDATION:\n")
    print(answer_text)
    print("=" * 80 + "\n")
    return answer_text

print("✅ RAG Generation Pipeline is ready for demonstration!")


## 🎯 Step 5: Demonstration Scenarios for Your Meeting

Run the cell below to see how the bot handles diverse, realistic queries:
1. **Scenario 1 (Dietary & Budget)**: Vegan + Gluten-Free dinner under $25.
2. **Scenario 2 (Executive Corporate)**: High-end quiet seafood dinner for client entertaining.
3. **Scenario 3 (Casual Family)**: Kid-friendly casual Mexican with outdoor seating.
4. **Scenario 4 (Cultural & Large Group)**: Certified Halal family dinner for 10 people.
5. **Scenario 5 (Hallucination Prevention Test)**: Asking for a city/cuisine not in the database to show strict grounding.


In [ ]:
# 🚀 Run Meeting Demonstration Scenarios

demo_scenarios = [
    # Scenario 1: Strict dietary & budget constraint
    "My friend is strictly vegan and also needs gluten-free options. We want a cozy dinner spot under $30 per person. What do you recommend?",
    
    # Scenario 2: Executive business & seafood
    "I need a quiet, upscale place with ocean views for an executive 6-person client dinner. We need top-quality seafood and an extensive wine list.",
    
    # Scenario 3: Casual, family & kids
    "Looking for a fun, casual Mexican place with an outdoor patio where we can bring kids and get great tacos and margaritas.",
    
    # Scenario 4: Halal & large group
    "We have a family gathering of 10 people and require 100% certified Halal food. Which restaurant would accommodate us best?",
    
    # Scenario 5: Negative / Out-of-catalog test (shows hallucination resistance)
    "Can you recommend a Chicago deep-dish pizza restaurant near Millennium Park?"
]

for scenario in demo_scenarios:
    ask_restaurant_bot(scenario)
    time.sleep(1) # Brief pause between scenarios to avoid hitting API rate spikes


## 🎤 Step 6: Interactive Live Q&A in the Meeting

Use the cell below during your meeting to let stakeholders ask their own custom questions! You can type any dietary requirement, cuisine preference, or occasion.


In [ ]:
# 💬 Interactive Query Cell (Try it live during your meeting!)
live_query = input("Ask ConciergeAI a dining question (or press Enter for default): ").strip()

if not live_query:
    live_query = "What is a great romantic spot downtown with handmade pasta and wine?"

ask_restaurant_bot(live_query)


## 📋 Summary & How to Extend This Bot

### Key Takeaways for the Meeting:
* **Domain Grounding**: The LLM relies on your proprietary documents, eliminating hallucinations.
* **Semantic Search**: Vector embeddings match user intent even with complex combinations of constraints (budget + diet + ambiance).
* **Source Transparency**: Every recommendation cites the exact restaurant name, neighborhood, and menu specifics from the catalog.

### How to Adapt to Other Use Cases:
To switch this bot to **Hospital Information**, **Travel Planner**, or **Car Recommendations**:
1. Replace `restaurant_documents` in **Step 1** with your own documents (or load from `.pdf`, `.csv`, `.json` using LangChain loaders).
2. Tweak the `PROMPT_TEMPLATE` persona in **Step 4**.
3. Re-run the notebook!
